In [ ]:
# Install required packages
!pip install fastapi uvicorn nest-asyncio pyngrok pandas fuzzywuzzy[speedup]

# Import required modules
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import pandas as pd
import nest_asyncio
from pyngrok import ngrok

import uvicorn

# Apply nest_asyncio to avoid event loop errors
nest_asyncio.apply()

# Create FastAPI app
app = FastAPI()

# Step 1: Mock Product Catalogue
data = [
    {
        "Assessment": "Cognitive Ability Test",
        "Skills": "Logical Reasoning, Math",
        "Job Role": "Analyst, Developer",
        "Level": "Entry",
        "Industry": "Tech, Finance",
        "Duration": "30 min"
    },
    {
        "Assessment": "Leadership Aptitude",
        "Skills": "Decision Making, Strategy",
        "Job Role": "Manager, Team Lead",
        "Level": "Mid",
        "Industry": "Any",
        "Duration": "45 min"
    },
    {
        "Assessment": "Sales Proficiency Test",
        "Skills": "Communication, Persuasion",
        "Job Role": "Sales Executive",
        "Level": "Entry",
        "Industry": "Retail, B2B",
        "Duration": "25 min"
    },
    {
        "Assessment": "Coding Simulation Test",
        "Skills": "Python, Algorithms",
        "Job Role": "Developer, Engineer",
        "Level": "Entry",
        "Industry": "Tech",
        "Duration": "60 min"
    },
    {
        "Assessment": "Project Management",
        "Skills": "Planning, Time Mgmt",
        "Job Role": "PM, Coordinator",
        "Level": "Mid",
        "Industry": "Tech, Infra",
        "Duration": "40 min"
    },
]

df = pd.DataFrame(data)

# Step 2: API input format
class UserInput(BaseModel):
    job_role: str
    level: str
    industry: str

# Step 3: Routes
@app.get("/")
def root():
    return {"message": "SHL Assessment Recommendation Engine is running"}

from fuzzywuzzy import fuzz
@app.post("/recommend")
def recommend(input: UserInput):
    job = input.job_role.lower().strip()
    level = input.level.lower().strip()
    industry = input.industry.lower().strip()

    # Optional: Normalize common synonyms
    if job == "software development":
        job = "developer"
    if industry == "technology":
        industry = "tech"

    def match(row):
        job_roles = str(row.get("Job Role", "")).split(",")
        industries = str(row.get("Industry", "")).split(",")
        level_val = str(row.get("Level", "")).lower()

        job_match = any(fuzz.token_sort_ratio(job, j.strip().lower()) >= 60 for j in job_roles)
        level_match = fuzz.token_sort_ratio(level, level_val) >= 60
        industry_match = any(fuzz.token_sort_ratio(industry, i.strip().lower()) >= 60 for i in industries)

        return job_match and level_match and industry_match

    matches = df[df.apply(match, axis=1)]

    if matches.empty:
        return {"message": "No close matches found. Try adjusting your input."}

    return {"recommendations": matches["Assessment"].tolist()}

!ngrok config add-authtoken 2wYPaTdGRKTb5HYVyR3WId4nyaz_2VanhFdEmQZs8YvcmhuyJ

ngrok.set_auth_token("2wYPaTdGRKTb5HYVyR3WId4nyaz_2VanhFdEmQZs8YvcmhuyJ")
public_url = ngrok.connect(8000)
print(f"Your app is live at: {public_url}/docs")

uvicorn.run(app, host="0.0.0.0", port=8000)



ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-85' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/main.py", line 580, in run
    server.run()
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run
    s

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Your app is live at: NgrokTunnel: "https://db4d-34-73-20-214.ngrok-free.app" -> "http://localhost:8000"/docs


INFO:     Started server process [2276]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-49' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/main.py", line 580, in run
    server.run()
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 66, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/nest_asyncio.py", line 92, in run_until_complete
   

INFO:     14.139.239.180:0 - "GET / HTTP/1.1" 200 OK
INFO:     14.139.239.180:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     14.139.239.180:0 - "GET /docs/ HTTP/1.1" 307 Temporary Redirect
INFO:     14.139.239.180:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     14.139.239.180:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     14.139.239.180:0 - "POST /recommend HTTP/1.1" 200 OK


In [11]:
from fuzzywuzzy import fuzz

user_job = "software development"
catalogue_job = "Developer"

score = fuzz.token_sort_ratio(user_job.lower(), catalogue_job.lower())

print(f"Fuzzy Score: {score}")

Fuzzy Score: 62
